In [1]:
!pip install alpaca-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 3.1 MB/s eta 0:00:00


### El Análisis de tu Estrategia

-cLa regla de 1 trade al día: Te obliga a ser extremadamente selectivo. Ya no buscas "cualquier cosa que se mueva", buscas el mejor setup del día. Si falla, apagas la pantalla y a otra cosa. Desarrollas una disciplina de hierro.

- El enfoque en el volumen: En rupturas de premarket, el volumen es el rey. Sin volumen, las rupturas se convierten en trampas para compradores (bull traps).

### Lo que debes vigilar (Alerta con el Capital):

Con un capital de 300 € a 1000 €, el "Stop-Loss fijo del 1%" requiere una mirada matemática estricta:

- Si tienes 500 € de capital, el 1% son 5 € de riesgo por operación.

- Si operas con acciones tradicionales, las comisiones del bróker se pueden comer tu beneficio y tu stop-loss. Por ejemplo, si entrar y salir te cuesta 2 € en total, ya estás perdiendo casi la mitad de tu riesgo permitido solo en comisiones.

**Solución para capital pequeño**: Para que este plan funcione con 300–1000 €, necesitas usar Micro-Futuros (como el Micro E-mini S&P 500, donde las comisiones son centavos) o un bróker de acciones/CFDs sin comisiones de ejecución (o muy bajas).

LLa forma más eficiente y segura es programar una Orden Bracket (**Bracket Order**) con un disparador de tipo Stop.

Esto significa que no necesitas dejar un bucle while True corriendo infinitamente en Colab (lo cual suele desconectarse). En su lugar, el script calcula el máximo del premercado/día, calcula tu stop-loss exacto del 1%, y envía la orden directamente a los servidores de Alpaca. El sistema ejecutará la compra automáticamente **solo si el precio rompe el máximo**, y colocará inmediatamente tu Stop-Loss y tu Take-Profit.

In [2]:
import datetime
from zoneinfo import ZoneInfo
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import MarketOrderRequest, StopOrderRequest, TakeProfitRequest, StopLossRequest
from alpaca.trading.enums import OrderSide, TimeInForce, OrderClass
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from google.colab import userdata

In [3]:
API_KEY = userdata.get('ALPACA_KEY')
SECRET_KEY = userdata.get('ALPACA_SECRET')
PAPER_TRADING = True  # Mantener en True para simulaciones

# Parámetros de la Estrategia
TICKER = "AAPL"          # Acción a operar (Alta liquidez/volumen: AAPL, TSLA, NVDA, AMD)
CAPITAL_MAX = 500       # Euros/Dólares asignados a esta operación (Entre 300 y 1000)
STOP_LOSS_PCT = 0.01     # 1% Fijo de Stop-Loss
TAKE_PROFIT_PCT = 0.02   # 2% de Objetivo (Relación Riesgo/Beneficio 1:2)

# Inicializar Clientes de Alpaca
trading_client = TradingClient(API_KEY, SECRET_KEY, paper=PAPER_TRADING)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

In [4]:
def obtener_maximo_del_dia(ticker):
    """Obtiene el precio máximo registrado hoy (incluyendo premarket corto)"""
    zona_ny = ZoneInfo("America/New_York")
    hoy = datetime.datetime.now(zona_ny).date()

    # Definimos el inicio del día en horario de Nueva York
    inicio_dia = datetime.datetime.combine(hoy, datetime.time(4, 0), tzinfo=zona_ny)
    fin_dia = datetime.datetime.combine(hoy, datetime.time(16, 0), tzinfo=zona_ny)

    request_params = StockBarsRequest(
        symbol_or_symbols=ticker,
        timeframe=TimeFrame.Minute,
        start=inicio_dia,
        end=fin_dia
    )

    bars = data_client.get_stock_bars(request_params)
    df = bars.df

    if df.empty:
        raise ValueError(f"No se encontraron datos para {ticker} hoy. ¿El mercado está abierto?")

    max_precio = df['high'].max()
    ultimo_precio = df['close'].iloc[-1]
    return float(max_precio), float(ultimo_precio)

In [5]:
def ejecutar_estrategia_micro_trading():
    # 1. Comprobar si ya tenemos posiciones abiertas hoy (Regla de 1 trade al día)
    posiciones = trading_client.get_all_positions()
    for pos in posiciones:
        if pos.symbol == TICKER:
            print(f"⚠️ Ya tienes una posición abierta en {TICKER}. Regla estricta: ¡1 trade al día!")
            return

    # 2. Obtener los precios clave
    try:
        max_del_dia, precio_actual = obtener_maximo_del_dia(TICKER)
        print(f"📈 {TICKER} - Precio Actual: ${precio_actual:.2f} | Máximo del Día: ${max_del_dia:.2f}")
    except Exception as e:
        print(f"❌ Error al obtener datos: {e}")
        return

    # Si el precio actual ya rompió el máximo por mucho, evitamos entrar tarde
    if precio_actual > max_del_dia * 1.005:
        print("⚠️ El precio ya se ha escapado demasiado del máximo. Operación cancelada para evitar FOMO.")
        return

    # 3. Calcular niveles de la orden
    # Colocamos la entrada ligeramente por encima del máximo para confirmar la ruptura (breakout)
    precio_entrada_trigger = round(max_del_dia + 0.02, 2)

    # Calcular niveles según porcentajes estricto de tu plan
    precio_stop_loss = round(precio_entrada_trigger * (1 - STOP_LOSS_PCT), 2)
    precio_take_profit = round(precio_entrada_trigger * (1 + TAKE_PROFIT_PCT), 2)

    # Calcular tamaño de la posición (Capital / Precio de entrada)
    cantidad_acciones = int(CAPITAL_MAX // precio_entrada_trigger)

    if cantidad_acciones == 0:
        print("❌ Capital insuficiente para comprar al menos 1 acción de este activo.")
        return

    print(f"🎯 Preparando Orden Bracket (Breakout):")
    print(f"   - Comprar: {cantidad_acciones} acciones de {TICKER}")
    print(f"   - Disparador de Entrada (Stop Price): ${precio_entrada_trigger:.2f}")
    print(f"   - 🛑 Stop-Loss (1%): ${precio_stop_loss:.2f}")
    print(f"   - 💰 Take-Profit (2%): ${precio_take_profit:.2f}")

    # 4. Construir y enviar la Orden Bracket
    # Usamos una orden Stop para que se active SOLO si el mercado sube y toca el precio_entrada_trigger
    orden_bracket = StopOrderRequest(
        symbol=TICKER,
        qty=cantidad_acciones,
        side=OrderSide.BUY,
        time_in_force=TimeInForce.DAY,
        stop_price=precio_entrada_trigger,
        order_class=OrderClass.BRACKET,
        take_profit=TakeProfitRequest(limit_price=precio_take_profit),
        stop_loss=StopLossRequest(stop_price=precio_stop_loss)
    )

    # Enviar orden al mercado
    try:
        envio = trading_client.submit_order(order_data=orden_bracket)
        print(f"✅ Orden Bracket enviada con éxito. ID: {envio.id}")
        print("Puedes apagar el entorno. Alpaca gestionará la entrada, el stop y el profit automáticamente.")
    except Exception as e:
        print(f"❌ Error al enviar la orden a Alpaca: {e}")

In [6]:
# Ejecutar el script
ejecutar_estrategia_micro_trading()

⚠️ Ya tienes una posición abierta en AAPL. Regla estricta: ¡1 trade al día!


### Cómo funciona este script bajo tus reglas:

- Filtro Automático de Disciplina: El script revisa trading_client.get_all_positions(). Si detecta que ya estás dentro de ese activo, frena la ejecución. Esto te blinda matemáticamente contra la tentación de promediar a la baja o reabrir operaciones por venganza.

- Uso de Orden Stop (Breakout Real): En lugar de comprar al precio de mercado actual, el script envía una orden condicional. Si fijas la entrada a $100.02$ (porque el máximo era $100$), la orden se quedará flotando en el servidor de Alpaca. Si el mercado nunca rompe el máximo y cae, tu dinero nunca se arriesga y la orden se cancela sola al fin del día.

- Gestión de Riesgo Automatizada (Bracket): En el mismo instante en que el precio toca tu punto de entrada, el servidor de Alpaca lanza en milisegundos dos órdenes de salida vinculadas: un Stop-Loss al $-1\%$ y un Take-Profit al $+2\%$. Si se ejecuta una, la otra se cancela automáticamente (One-Cancels-Other).

Ejecuta este script **entre las 15:15 y las 15:30 (Hora de España / CET)**, justo antes o durante los primeros minutos de la apertura de Nueva York (15:30 CET). Así el script leerá con precisión el volumen y el rango máximo que se formó durante el premercado americano.